In [598]:
#1 The Data

In [599]:
import torch
from torch_geometric.datasets import MovieLens100K

In [600]:
dataset = MovieLens100K(root = "./data")

In [601]:
#data contains 1 graph
len(dataset)

1

In [698]:
#graph info
data = dataset[0]
print(data)

HeteroData(
  movie={ x=[1682, 18] },
  user={ x=[943, 24] },
  (user, rates, movie)={
    edge_index=[2, 80000],
    rating=[80000],
    time=[80000],
    edge_label_index=[2, 20000],
    edge_label=[20000],
  },
  (movie, rated_by, user)={
    edge_index=[2, 80000],
    rating=[80000],
    time=[80000],
  }
)


In [603]:
print(data.node_types)

['movie', 'user']


In [604]:
print(data.edge_types)

[('user', 'rates', 'movie'), ('movie', 'rated_by', 'user')]


In [605]:
#movie vectors represent movie genres (one movie can belong to several genres).
movie_x = data["movie"].x
print(movie_x[:5])

tensor([[0., 0., 1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.],
        [1., 0., 0., 0., 1., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 1., 0., 1., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.]])


In [606]:
#2 GCN baseline

In [607]:
#2.1. Transform edge index for GCN
#Shift movie indices and unite them with user indices in one raw: 

In [608]:
edge_index = data["user", "rates", "movie"].edge_index
print(edge_index)

tensor([[   0,    0,    0,  ...,  942,  942,  942],
        [   0,    1,    2,  ..., 1187, 1227, 1329]])


In [609]:
movie_index = edge_index[1]
print(movie_index)

tensor([   0,    1,    2,  ..., 1187, 1227, 1329])


In [610]:
number_users = data["user"].num_nodes
print(number_users)

943


In [611]:
transformed_movie_index = movie_index + number_users
print(transformed_movie_index)

tensor([ 943,  944,  945,  ..., 2130, 2170, 2272])


In [612]:
user_index = edge_index[0]
print(user_index)

tensor([  0,   0,   0,  ..., 942, 942, 942])


In [613]:
pos_edge_index = torch.stack([user_index, transformed_movie_index], dim=0)
print(pos_edge_index)

tensor([[   0,    0,    0,  ...,  942,  942,  942],
        [ 943,  944,  945,  ..., 2130, 2170, 2272]])


In [614]:
#2.2. Sample negative examples

In [615]:
number_movies = data["movie"].num_nodes
print(number_movies)

1682


In [616]:
number_pos = data["user", "rates", "movie"].edge_index.shape[1]
print(number_pos)

80000


In [617]:
from torch_geometric.utils import negative_sampling
neg_edge_index = negative_sampling(edge_index, (number_users, number_movies), number_pos)
print(neg_edge_index)
print(neg_edge_index.shape)

tensor([[ 655,  613,  414,  ...,   59,  342,  331],
        [ 644,  588,  990,  ..., 1674, 1503, 1031]])
torch.Size([2, 80000])


In [618]:
neg_movie_index = neg_edge_index[1]
print(neg_movie_index)

tensor([ 644,  588,  990,  ..., 1674, 1503, 1031])


In [619]:
transform_neg_movie_index = neg_movie_index + number_users
print(transform_neg_movie_index)

tensor([1587, 1531, 1933,  ..., 2617, 2446, 1974])


In [620]:
neg_user_index = neg_edge_index[0]
print(neg_user_index)

tensor([655, 613, 414,  ...,  59, 342, 331])


In [621]:
neg_edge_index = torch.stack([neg_user_index, transform_neg_movie_index], dim=0)
print(neg_edge_index)

tensor([[ 655,  613,  414,  ...,   59,  342,  331],
        [1587, 1531, 1933,  ..., 2617, 2446, 1974]])


In [622]:
#2.3. Create training data

In [623]:
training_edge_index = torch.cat([pos_edge_index, neg_edge_index], dim=1)
print(training_edge_index)
print(training_edge_index.shape)

tensor([[   0,    0,    0,  ...,   59,  342,  331],
        [ 943,  944,  945,  ..., 2617, 2446, 1974]])
torch.Size([2, 160000])


In [624]:
y_1 = torch.ones(pos_edge_index.shape[1])
print(y_1)
print(y_1.shape)
y_2 = torch.zeros(neg_edge_index.shape[1])
print(y_2)
print(y_2.shape)

tensor([1., 1., 1.,  ..., 1., 1., 1.])
torch.Size([80000])
tensor([0., 0., 0.,  ..., 0., 0., 0.])
torch.Size([80000])


In [625]:
y_train = torch.cat([y_1, y_2])
print(y_train.shape)

torch.Size([160000])


In [626]:
#2.4. Create GCN Recommender

In [627]:
feature_users = data["user"].x.shape[1]
print(feature_users)

24


In [628]:
feature_movies = data["movie"].x.shape[1]
print(feature_movies)

18


In [629]:
import torch.nn.functional as F
from torch import nn
from torch_geometric.nn import GCNConv

In [630]:
class GCNModel(nn.Module):
    def __init__(self, init_dim_users,
                 init_dim_movies,
                 dim_unified,
                 hidden_dim,
                 embed_dim, 
                 prediction_binary):
        super().__init__()
        #Your code goes here#
        self.transform_users = nn.Linear(init_dim_users, dim_unified)
        self.transform_movies = nn.Linear(init_dim_movies, dim_unified)
        self.gcn1 = GCNConv(dim_unified, hidden_dim)
        self.gcn2 = GCNConv(hidden_dim, embed_dim)
        self.head = nn.Linear(embed_dim * 2, prediction_binary)
    def forward(self, users, movies, message_pass_adj, training_adj):
        #Your code goes here#
        u = self.transform_users(users)
        m = self.transform_movies(movies)
        x = torch.cat([u, m], dim=0)
        x = self.gcn1.forward(x, message_pass_adj)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.gcn2.forward(x, message_pass_adj)
        x = F.dropout(x, p=0.5, training=self.training)
        x = torch.cat([ x[training_adj[0]], x[training_adj[1]] ], dim=1)
        x = self.head(x)
        return x

In [631]:
#2.5. Train Recommender

In [632]:
model = GCNModel(init_dim_users=feature_users, init_dim_movies=feature_movies, dim_unified=32, hidden_dim=64, embed_dim=32, prediction_binary=1)
optim = torch.optim.Adam(model.parameters(), 0.01)
loss = nn.BCEWithLogitsLoss()

In [633]:
for epoch in range(201):
    # Predict the training instances, compute the error, backpropagate the error and update the model weights
    pred = model.forward(users=data["user"].x, movies=data["movie"].x, message_pass_adj=pos_edge_index, training_adj=training_edge_index)
    l = loss(pred.squeeze(1), y_train)
    optim.zero_grad()
    l.backward()
    optim.step()
    if epoch%20 == 0:
        predictions = (pred.sigmoid() >= 0.5).float()
        accuracy = (predictions.squeeze(1) == y_train).float().mean()
        print(f"epoch {epoch}, CEL: {l}, accuracy: {accuracy}")

epoch 0, CEL: 0.6836556792259216, accuracy: 0.5764687657356262
epoch 20, CEL: 0.5603476762771606, accuracy: 0.7272499799728394
epoch 40, CEL: 0.5229414105415344, accuracy: 0.7392187714576721
epoch 60, CEL: 0.5162206888198853, accuracy: 0.7432437539100647
epoch 80, CEL: 0.5117242932319641, accuracy: 0.7504812479019165
epoch 100, CEL: 0.5104807615280151, accuracy: 0.7490624785423279
epoch 120, CEL: 0.5037199258804321, accuracy: 0.7554187774658203
epoch 140, CEL: 0.50510573387146, accuracy: 0.7533312439918518
epoch 160, CEL: 0.50626540184021, accuracy: 0.7488625049591064
epoch 180, CEL: 0.5037729740142822, accuracy: 0.7558500170707703
epoch 200, CEL: 0.5021785497665405, accuracy: 0.7531625032424927


In [634]:
#2.6 Evaluate Recommender

In [635]:
#2.6.1 Transform edge index for GCN
#Shift movie indices and unite them with user indices in one raw:

In [636]:
edge_label_index = data["user", "rates", "movie"].edge_label_index
print(edge_label_index)

tensor([[  0,   0,   0,  ..., 458, 459, 461],
        [  5,   9,  11,  ..., 933,   9, 681]])


In [637]:
movie_test_index = edge_label_index[1]
print(movie_test_index)

tensor([  5,   9,  11,  ..., 933,   9, 681])


In [638]:
print(number_users)

943


In [639]:
transform_movie_test_index = movie_test_index + number_users
print(transform_movie_test_index)

tensor([ 948,  952,  954,  ..., 1876,  952, 1624])


In [640]:
user_test_index = (data["user", "rates", "movie"].edge_label_index)[0]
print(user_test_index)

tensor([  0,   0,   0,  ..., 458, 459, 461])


In [641]:
pos_edge_test_index = torch.stack([user_test_index, transform_movie_test_index], dim=0)
print(pos_edge_test_idx)

tensor([[   0,    0,    0,  ...,  458,  459,  461],
        [ 948,  952,  954,  ..., 1876,  952, 1624]])


In [642]:
#2.6.2 Sample negative examples

In [643]:
print(number_users)

943


In [644]:
print(number_movies)

1682


In [645]:
number_pos_test = data["user", "rates", "movie"].edge_label_index.shape[1]
print(number_pos_test)

20000


In [646]:
from torch_geometric.utils import negative_sampling
neg_edge_test_index = negative_sampling(edge_label_index, (number_users, number_movies), number_pos_test)
print(neg_edge_test_index)
print(neg_edge_test_index.shape)

tensor([[ 828,  803,  427,  ...,    1,  764,  887],
        [1542,  363,   75,  ...,  970,  757,   25]])
torch.Size([2, 20000])


In [647]:
transform_movie_neg_edge_test_index = neg_edge_test_index[1] + number_users
print(transform_movie_neg_edge_test_index)

tensor([2485, 1306, 1018,  ..., 1913, 1700,  968])


In [648]:
user_neg_edge_test_index = neg_edge_test_index[0]
print(user_neg_edge_test_index)

tensor([828, 803, 427,  ...,   1, 764, 887])


In [649]:
neg_edge_test_index = torch.stack([user_neg_edge_test_index, transform_movie_neg_edge_test_index], dim=0)
print(neg_edge_test_index)

tensor([[ 828,  803,  427,  ...,    1,  764,  887],
        [2485, 1306, 1018,  ..., 1913, 1700,  968]])


In [650]:
#2.6.2 Create test data

In [651]:
test_edge_index = torch.cat([pos_edge_test_index, neg_edge_test_index], dim=1)
print(test_edge_index)
print(test_edge_index.shape)

tensor([[   0,    0,    0,  ...,    1,  764,  887],
        [ 948,  952,  954,  ..., 1913, 1700,  968]])
torch.Size([2, 40000])


In [652]:
y_test_1 = torch.ones(pos_edge_test_idx.shape[1])
print(y_1)
print(y_1.shape)
y_test_2 = torch.zeros(neg_edge_test_idx.shape[1])
print(y_2)
print(y_2.shape)

tensor([1., 1., 1.,  ..., 1., 1., 1.])
torch.Size([80000])
tensor([0., 0., 0.,  ..., 0., 0., 0.])
torch.Size([80000])


In [653]:
y_test = torch.cat([y_test_1, y_test_2])
print(y_test.shape)

torch.Size([40000])


In [654]:
#2.6.3. Launch evaluation

In [655]:
model.eval()
with torch.no_grad():
    pred_test = model.forward(users=data["user"].x, movies=data["movie"].x, message_pass_adj=pos_edge_test_index, training_adj=test_edge_index)
    l_test = loss(pred_test.squeeze(1), y_test)
    predictions_test = (pred_test.sigmoid() >= 0.5).float()
    accuracy_test = (predictions_test.squeeze(1) == y_test).float().mean()
    print(f"test CEL: {l_test}, test accuracy: {accuracy_test}")

test CEL: 0.691581666469574, test accuracy: 0.5494250059127808


In [656]:
#3. R-GCN

In [700]:
print(data)

HeteroData(
  movie={ x=[1682, 18] },
  user={ x=[943, 24] },
  (user, rates, movie)={
    edge_index=[2, 80000],
    rating=[80000],
    time=[80000],
    edge_label_index=[2, 20000],
    edge_label=[20000],
  },
  (movie, rated_by, user)={
    edge_index=[2, 80000],
    rating=[80000],
    time=[80000],
  }
)


In [658]:
#3.1. Set arguments for R-GCN layer

In [659]:
movie = data["movie"].x
print(movie)

tensor([[0., 0., 1.,  ..., 0., 0., 0.],
        [1., 1., 0.,  ..., 1., 0., 0.],
        [0., 0., 0.,  ..., 1., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])


In [660]:
user = data["user"].x

In [661]:
pos_edge_index_relational = data["user", "rates", "movie"].edge_index
print(pos_edge_index_relational)

tensor([[   0,    0,    0,  ...,  942,  942,  942],
        [   0,    1,    2,  ..., 1187, 1227, 1329]])


In [662]:
relations = data["user", "rates", "movie"].rating
print(relations)

tensor([5, 3, 4,  ..., 3, 3, 3])


In [663]:
print(genre_ids)
print(genre_ids.shape)

tensor([ 2,  3,  4,  ..., 13,  4,  7])
torch.Size([2891])


In [664]:
#3.2. Create relational training data

In [665]:
print(training_edge_index)

tensor([[   0,    0,    0,  ...,   59,  342,  331],
        [ 943,  944,  945,  ..., 2617, 2446, 1974]])


In [666]:
transform_back = training_edge_index[1] - number_users
print(transform_back)

tensor([   0,    1,    2,  ..., 1674, 1503, 1031])


In [667]:
training_users = training_edge_index[0]
print(training_users)

tensor([  0,   0,   0,  ...,  59, 342, 331])


In [668]:
training_edge_index_relational = torch.stack([training_users, transform_back], dim=0)
print(training_edge_index_relational)

tensor([[   0,    0,    0,  ...,   59,  342,  331],
        [   0,    1,    2,  ..., 1674, 1503, 1031]])


In [669]:
y_train_relational = y_train
print(y_train_relational)
print(y_train.shape)

tensor([1., 1., 1.,  ..., 0., 0., 0.])
torch.Size([160000])


In [670]:
#3.3. Create RGCN Recommender

In [671]:
from torch_geometric.nn import RGCNConv

In [672]:
feature_users = data["user"].x.shape[1]
print(feature_users)

24


In [673]:
feature_movies = data["movie"].x.shape[1]
print(feature_movies)

18


In [674]:
class RGCNModel(nn.Module):
    def __init__(self, dim_users,
                 dim_movies,
                 hidden_dim,
                 embed_dim, 
                 prediction_binary):
        super().__init__()
        #Your code goes here#
        self.rgcn1 = RGCNConv((dim_users, dim_movies), hidden_dim, num_relations=5+1)
        self.rgcn2 = RGCNConv((dim_users, hidden_dim), embed_dim, num_relations=5+1)
        self.head = nn.Linear(embed_dim + feature_users, prediction_binary)
    def forward(self, user, movie, message_pass_adj_relational, relation_types, training_adj_relational):
        #Your code goes here#
        m_emb = self.rgcn1.forward((user, movie), message_pass_adj_relational, relation_types)
        m_emb = F.relu(m_emb)
        m_emb = F.dropout(m_emb, p=0.5, training=self.training)
        m_emb = self.rgcn2.forward((user, m_emb), message_pass_adj_relational, relation_types)
        m_emb = F.dropout(m_emb, p=0.5, training=self.training)
        x = torch.cat([ user[training_adj_relational[0]], m_emb[training_adj_relational[1]] ], dim=1)
        x = self.head(x)
        return x

In [675]:
#3.4. Train RGCN

In [676]:
model_rgcn = RGCNModel(dim_users=feature_users, dim_movies=feature_movies, hidden_dim=64, embed_dim=32, prediction_binary=1)
optim = torch.optim.Adam(model_rgcn.parameters(), 0.01)
loss = nn.BCEWithLogitsLoss()

In [706]:
print(model_rgcn.parameters)

<bound method Module.parameters of RGCNModel(
  (rgcn1): RGCNConv((24, 18), 64, num_relations=6)
  (rgcn2): RGCNConv((24, 64), 32, num_relations=6)
  (head): Linear(in_features=56, out_features=1, bias=True)
)>


In [707]:
for name, p in model.named_parameters():
    print(name, p.requires_grad)

transform_users.weight True
transform_users.bias True
transform_movies.weight True
transform_movies.bias True
gcn1.bias True
gcn1.lin.weight True
gcn2.bias True
gcn2.lin.weight True
head.weight True
head.bias True


In [677]:
for epoch in range(201):
    # Predict the training instances, compute the error, backpropagate the error and update the model weights
    pred = model_rgcn.forward(user=user, movie=movie, message_pass_adj_relational=pos_edge_index_relational,
                              relation_types=relations, training_adj_relational=training_edge_index_relational)
    l = loss(pred.squeeze(1), y_train_relational)
    optim.zero_grad()
    l.backward()
    optim.step()
    if epoch%20 == 0:
        predictions = (pred.sigmoid() >= 0.5).float()
        accuracy = (predictions.squeeze(1) == y_train).float().mean()
        print(f"epoch {epoch}, CEL: {l}, accuracy: {accuracy}")

epoch 0, CEL: 0.7161396741867065, accuracy: 0.4925312399864197
epoch 20, CEL: 0.5550726056098938, accuracy: 0.7219125032424927
epoch 40, CEL: 0.5310304760932922, accuracy: 0.7388499975204468
epoch 60, CEL: 0.5218316912651062, accuracy: 0.7431437373161316
epoch 80, CEL: 0.5159943699836731, accuracy: 0.7469375133514404
epoch 100, CEL: 0.5161190629005432, accuracy: 0.7483875155448914
epoch 120, CEL: 0.5112212896347046, accuracy: 0.750712513923645
epoch 140, CEL: 0.5120565295219421, accuracy: 0.7509437203407288
epoch 160, CEL: 0.5097905993461609, accuracy: 0.7511125206947327
epoch 180, CEL: 0.5079678893089294, accuracy: 0.7523937225341797
epoch 200, CEL: 0.5080640912055969, accuracy: 0.7537000179290771


In [678]:
#3.5. Evaluate RGCN Recommender

In [679]:
test_pos_edge_index_relational = data["user", "rates", "movie"].edge_label_index
print(test_pos_edge_index_relational)
print(test_pos_edge_index_relational.shape)

tensor([[  0,   0,   0,  ..., 458, 459, 461],
        [  5,   9,  11,  ..., 933,   9, 681]])
torch.Size([2, 20000])


In [680]:
test_relations = data["user", "rates", "movie"].edge_label.long()
print(test_relations)

tensor([5, 3, 5,  ..., 3, 3, 5])


In [681]:
print(test_edge_index)
print(test_edge_index.shape)

tensor([[   0,    0,    0,  ...,    1,  764,  887],
        [ 948,  952,  954,  ..., 1913, 1700,  968]])
torch.Size([2, 40000])


In [682]:
test_transform_back = test_edge_index[1] - number_users
print(test_transform_back)

tensor([  5,   9,  11,  ..., 970, 757,  25])


In [683]:
test_users = test_edge_index[0]
print(test_users)

tensor([  0,   0,   0,  ...,   1, 764, 887])


In [684]:
test_edge_index_relational = torch.stack([test_users, test_transform_back], dim=0)
print(test_edge_index_relational)
print(test_edge_index_relational.shape)

tensor([[  0,   0,   0,  ...,   1, 764, 887],
        [  5,   9,  11,  ..., 970, 757,  25]])
torch.Size([2, 40000])


In [685]:
y_test_relational = y_test
print(y_test_relational)
print(y_test.shape)

tensor([1., 1., 1.,  ..., 0., 0., 0.])
torch.Size([40000])


In [686]:
#3.6. Launch Evaluation

In [687]:
model_rgcn.eval()
with torch.no_grad():
    pred_test_relational = model_rgcn.forward(user=user, movie=movie, message_pass_adj_relational=test_pos_edge_index_relational,
                              relation_types=test_relations, training_adj_relational=test_edge_index_relational)
    l_test_relational = loss(pred_test_relational.squeeze(1), y_test_relational)
    predictions_test_relational = (pred_test_relational.sigmoid() >= 0.5).float()
    accuracy_test_relational = (predictions_test_relational.squeeze(1) == y_test).float().mean()
    print(f"test CEL: {l_test_relational}, test accuracy: {accuracy_test_relational}")

test CEL: 0.6147826910018921, test accuracy: 0.6748999953269958


In [688]:
#4 RCGN + genres: upgrade the RGCN recommender with genre relations

In [689]:
#4.1. Create Genre Relation

In [764]:
#since there as no edge relation in the graph, we need to reconstruct it from the movie embeddings.
#with torch nonzero define the exact place of each "movie-genre" correspondence in movie_x 
movie_ids, genre_ids = movie_x.nonzero(as_tuple=True)
#create a new edge type in the data
data["movie", "has_genre", "genre"].edge_index = torch.stack([genre_ids, movie_ids], dim=0)
print(data)

HeteroData(
  movie={ x=[1682, 18] },
  user={ x=[943, 24] },
  genre={ num_nodes=18 },
  (user, rates, movie)={
    edge_index=[2, 80000],
    rating=[80000],
    time=[80000],
    edge_label_index=[2, 20000],
    edge_label=[20000],
  },
  (movie, rated_by, user)={
    edge_index=[2, 80000],
    rating=[80000],
    time=[80000],
  },
  (movie, has_genre, genre)={ edge_index=[2, 2891] }
)


In [775]:
genre_idx = data["movie", "has_genre", "genre"].edge_index
print(genre_idx)

tensor([[   2,    3,    4,  ...,   13,    4,    7],
        [   0,    0,    0,  ..., 1679, 1680, 1681]])


In [777]:
train_genre_idx = genre_idx[:, :2312]
print(train_genre_idx)

tensor([[   2,    3,    4,  ...,    7,    7,    7],
        [   0,    0,    0,  ..., 1264, 1265, 1266]])


In [782]:
test_genre_idx = genre_idx[:, 2312:]
print(test_genre_idx)

tensor([[   7,    4,   13,  ...,   13,    4,    7],
        [1267, 1268, 1268,  ..., 1679, 1680, 1681]])


In [786]:
data["movie", "has_genre", "genre"].train = train_genre_idx
print(data)

HeteroData(
  movie={ x=[1682, 18] },
  user={ x=[943, 24] },
  genre={ num_nodes=18 },
  (user, rates, movie)={
    edge_index=[2, 80000],
    rating=[80000],
    time=[80000],
    edge_label_index=[2, 20000],
    edge_label=[20000],
  },
  (movie, rated_by, user)={
    edge_index=[2, 80000],
    rating=[80000],
    time=[80000],
  },
  (movie, has_genre, genre)={
    edge_index=[2, 2891],
    train=[2, 2312],
  }
)


In [787]:
data["movie", "has_genre", "genre"].test = test_genre_idx
print(data)

HeteroData(
  movie={ x=[1682, 18] },
  user={ x=[943, 24] },
  genre={ num_nodes=18 },
  (user, rates, movie)={
    edge_index=[2, 80000],
    rating=[80000],
    time=[80000],
    edge_label_index=[2, 20000],
    edge_label=[20000],
  },
  (movie, rated_by, user)={
    edge_index=[2, 80000],
    rating=[80000],
    time=[80000],
  },
  (movie, has_genre, genre)={
    edge_index=[2, 2891],
    train=[2, 2312],
    test=[2, 579],
  }
)


In [691]:
#4.2 Create genre nodes

In [765]:
num_genres = data["movie"].x.size(1)
print(num_genres)

18


In [773]:
num_genres = torch.arange(num_genres)
print(num_genres)

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17])


In [767]:
num_genres = data["movie"].x.size(1)  # 18
data["genre"].num_nodes = num_genres
print(data)

HeteroData(
  movie={ x=[1682, 18] },
  user={ x=[943, 24] },
  genre={ num_nodes=18 },
  (user, rates, movie)={
    edge_index=[2, 80000],
    rating=[80000],
    time=[80000],
    edge_label_index=[2, 20000],
    edge_label=[20000],
  },
  (movie, rated_by, user)={
    edge_index=[2, 80000],
    rating=[80000],
    time=[80000],
  },
  (movie, has_genre, genre)={ edge_index=[2, 2891] }
)


In [788]:
print(train_genre_idx)

tensor([[   2,    3,    4,  ...,    7,    7,    7],
        [   0,    0,    0,  ..., 1264, 1265, 1266]])


In [791]:
train_genre_labels = train_genre_idx[0]
print(train_genre_labels)

tensor([2, 3, 4,  ..., 7, 7, 7])


In [797]:
#4.3 Create RGCN-genre recommender

In [792]:
class RGCNModel_genre(nn.Module):
    def __init__(self, dim_users,
                 dim_movies,
                 dim_genres,
                 hidden_dim,
                 embed_dim, 
                 prediction_binary):
        super().__init__()
        #Your code goes here#
        self.genre = torch.nn.Embedding(num_embeddings=dim_movies, embedding_dim=dim_genres)
        self.rgcn1 = RGCNConv((dim_users, dim_movies), hidden_dim, num_relations=5+1)
        self.rgcn2 = RGCNConv((dim_genres, dim_movies), hidden_dim, num_relations=18)
        self.rgcn3 = RGCNConv((dim_users, hidden_dim), embed_dim, num_relations=5+1)
        self.rgcn4 = RGCNConv((dim_genres, hidden_dim), embed_dim, num_relations=18)
        self.head = nn.Linear(embed_dim + feature_users, prediction_binary)
    def forward(self,
                user, movie, genre,
                message_pass_adj_relational_user_movie, message_pass_adj_relational_genre_movie,
                relation_types_ratings, relation_types_genres,
                training_adj_relational):
        #Your code goes here#
        m_emb_1 = self.rgcn1.forward((user, movie), message_pass_adj_relational_user_movie, relation_types_ratings)
        genre = self.genre(genre)
        m_emb_2 = self.rgcn2.forward((genre, movie), message_pass_adj_relational_genre_movie, relation_types_genres)
        m_emb = m_emb_1 + m_emb_2
        m_emb = F.relu(m_emb)
        m_emb = F.dropout(m_emb, p=0.5, training=self.training)
        m_emb_1 = self.rgcn3.forward((user, m_emb), message_pass_adj_relational_user_movie, relation_types_ratings)
        m_emb_2 = self.rgcn4.forward((genre, m_emb), message_pass_adj_relational_genre_movie, relation_types_genres)
        m_emb = m_emb_1 + m_emb_2
        m_emb = F.dropout(m_emb, p=0.5, training=self.training)
        x = torch.cat([ user[training_adj_relational[0]], m_emb[training_adj_relational[1]] ], dim=1)
        x = self.head(x)
        return x

In [793]:
model_rgcn_genre = RGCNModel_genre(dim_users=feature_users, dim_movies=feature_movies, dim_genres=30,
                                   hidden_dim=64, embed_dim=32, prediction_binary=1)
optim = torch.optim.Adam(model_rgcn_genre.parameters(), 0.01)
loss = nn.BCEWithLogitsLoss()

In [798]:
#4.4. Train RGCN-genre recommender

In [795]:
for epoch in range(201):
    # Predict the training instances, compute the error, backpropagate the error and update the model weights
    pred = model_rgcn_genre.forward(user=user, movie=movie, genre=num_genres,
                                    message_pass_adj_relational_user_movie=pos_edge_index_relational,
                                    message_pass_adj_relational_genre_movie=train_genre_idx,
                                    relation_types_ratings=relations, relation_types_genres=train_genre_labels,
                                    training_adj_relational=training_edge_index_relational)
    l = loss(pred.squeeze(1), y_train_relational)
    optim.zero_grad()
    l.backward()
    optim.step()
    if epoch%20 == 0:
        predictions = (pred.sigmoid() >= 0.5).float()
        accuracy = (predictions.squeeze(1) == y_train).float().mean()
        print(f"epoch {epoch}, CEL: {l}, accuracy: {accuracy}")

epoch 0, CEL: 0.9357476830482483, accuracy: 0.47815001010894775
epoch 20, CEL: 0.6258471608161926, accuracy: 0.6559125185012817
epoch 40, CEL: 0.5898661613464355, accuracy: 0.6877687573432922
epoch 60, CEL: 0.5673991441726685, accuracy: 0.7067375183105469
epoch 80, CEL: 0.5529540181159973, accuracy: 0.7194437384605408
epoch 100, CEL: 0.542675793170929, accuracy: 0.727400004863739
epoch 120, CEL: 0.5363293290138245, accuracy: 0.7334750294685364
epoch 140, CEL: 0.5330056548118591, accuracy: 0.7363749742507935
epoch 160, CEL: 0.5332180857658386, accuracy: 0.7361249923706055
epoch 180, CEL: 0.5284926891326904, accuracy: 0.7387437224388123
epoch 200, CEL: 0.5257536172866821, accuracy: 0.7419437766075134


In [799]:
#4.5. Evaluate RGCN-genre recommender

In [800]:
print(test_genre_idx)

tensor([[   7,    4,   13,  ...,   13,    4,    7],
        [1267, 1268, 1268,  ..., 1679, 1680, 1681]])


In [801]:
test_genre_labels = test_genre_idx[0]
print(test_genre_labels)

tensor([ 7,  4, 13,  4,  4,  4,  7,  7, 15, 14, 15, 15,  7,  0,  5,  7, 11,  1,
         3,  5,  7,  7,  4,  7, 12,  7,  4, 11, 13,  4,  4, 13,  7, 13,  4,  3,
         8,  1,  3,  8, 14,  6,  4,  7,  4,  7,  7, 13,  4, 11,  7, 13,  7, 13,
         4,  4,  0,  0,  5,  4,  7,  6,  4,  7,  7,  7,  4,  7,  4,  7,  9, 12,
        15,  0,  1, 15,  7, 13,  7,  7,  6,  7,  7,  4,  4,  7,  7, 15,  7,  7,
         7,  7,  7,  7,  6,  7,  4,  7,  7,  7,  3,  4,  8,  4,  7,  4,  7, 13,
         7,  7,  7,  7,  7,  7,  7,  7,  4,  7,  4,  7, 13,  7,  7,  4, 13, 16,
         0,  7,  4,  4,  0,  6,  0,  0,  7,  6,  7,  2,  7,  7,  7, 15,  4, 10,
         0,  4,  7,  4,  4, 13,  6, 13,  7,  7,  7,  1,  3,  7,  4, 13, 14, 13,
         7,  7,  5,  7,  7,  0, 15, 13,  7,  7,  7,  7, 15,  7, 13,  7,  7,  4,
         7,  4,  7,  7, 13,  0,  4,  2,  3,  7,  1, 14,  2,  3,  0,  0,  0,  3,
         0, 14,  7,  7,  0, 14,  4,  7,  9, 15,  7, 16,  4,  7, 13,  4,  7,  4,
         7,  7,  4,  7, 13,  7, 15,  7, 

In [803]:
model_rgcn_genre.eval()
with torch.no_grad():
    pred_test_relational_genre = model_rgcn_genre.forward(user=user, movie=movie, genre=num_genres,
                                                          message_pass_adj_relational_user_movie=test_pos_edge_index_relational,
                                                          message_pass_adj_relational_genre_movie=test_genre_idx,
                                                          relation_types_ratings=test_relations, relation_types_genres=test_genre_labels,
                                                          training_adj_relational=test_edge_index_relational)
    l_test_relational_genre = loss(pred_test_relational_genre.squeeze(1), y_test_relational)
    predictions_test_relational_genre = (pred_test_relational_genre.sigmoid() >= 0.5).float()
    accuracy_test_relational_genre = (predictions_test_relational_genre.squeeze(1) == y_test).float().mean()
    print(f"test CEL: {l_test_relational}, test accuracy: {accuracy_test_relational}")

test CEL: 0.6147826910018921, test accuracy: 0.6748999953269958
